In [1]:
!pip install scikit-learn pandas numpy scipy

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix

emails_data = [
    ("Urgent! Your account has been suspended. Click here to verify: http://paypal-security-alert.xyz/login", "phishing"),
    ("Congratulations! You won $1,000,000. Click now to claim: http://prize-winner.tk/claim", "phishing"),
    ("Your bank account requires immediate verification. Login: http://secure-bank-update.ml/verify", "phishing"),
    ("ALERT: Suspicious activity detected. Reset password: http://amazon-security.gq/reset", "phishing"),
    ("Your PayPal account is limited. Verify identity at: http://paypal-verify-now.ga/confirm", "phishing"),
    ("IRS Notice: You have unclaimed tax refund. Click: http://irs-refund-claim.ml/apply", "phishing"),
    ("Your Apple ID was used in another country. Secure now: http://apple-id-protect.tk/secure", "phishing"),
    ("WINNER! You have been selected for $500 gift card. Claim: http://gift-card-winner.gq/claim", "phishing"),
    ("Your password expires today! Update immediately: http://microsoft-password-update.cf/renew", "phishing"),
    ("Your credit card was charged $999. Dispute now: http://dispute-charge-now.tk/cancel", "phishing"),
    ("DHL: Your package is held. Pay customs fee: http://dhl-customs-fee.ga/pay", "phishing"),
    ("Free iPhone 15! Limited offer ends soon. Click: http://free-iphone-offer.tk/get", "phishing"),
    ("Your Google account will be terminated. Act now: http://google-account-save.cf/verify", "phishing"),
    ("Final warning: Unpaid invoice. Pay now: http://invoice-payment-urgent.gq/pay", "phishing"),
    ("Security breach detected in your account. Fix: http://account-security-fix.tk/update", "phishing"),
    ("Dear customer verify your information to avoid suspension http://fake-bank-verify.xyz/login now", "phishing"),
    ("Click here to get your free gift card reward limited time http://reward-claim-now.ga/gift", "phishing"),
    ("Your social security number has been compromised verify http://ssa-verify-now.ml/ssn", "phishing"),
    ("You have won lottery prize money claim your winnings http://lottery-winner-prize.cf/claim", "phishing"),
    ("Account verification required immediately or access blocked http://verify-account-now.gq/id", "phishing"),
    ("Hi John, please find attached the meeting agenda for tomorrow's project review.", "safe"),
    ("Your monthly bank statement is ready. Login at yourbank.com to view.", "safe"),
    ("Thank you for your purchase! Your order #12345 has been confirmed.", "safe"),
    ("Team meeting scheduled for Monday at 10 AM. Conference room B.", "safe"),
    ("Your subscription has been renewed successfully. Thank you for staying with us.", "safe"),
    ("Please review the attached quarterly report and share your feedback.", "safe"),
    ("Reminder: Your dentist appointment is on Friday at 3 PM.", "safe"),
    ("Hi, I wanted to follow up on our conversation from last week.", "safe"),
    ("Your password was successfully changed. If you did not do this contact support.", "safe"),
    ("Welcome to our newsletter! Here are this week's top articles.", "safe"),
    ("Invoice #INV-2024-001 for services rendered in November is attached.", "safe"),
    ("Your flight booking confirmation: Delhi to Mumbai on Dec 15.", "safe"),
    ("Please complete the feedback form for our recent service.", "safe"),
    ("Your job application has been received. We will contact you shortly.", "safe"),
    ("The project deadline has been extended to next Friday. Please plan accordingly.", "safe"),
    ("Your package has been shipped. Track at fedex.com with code 789456.", "safe"),
    ("Reminder to submit your timesheet by end of day Friday.", "safe"),
    ("Hi team please join us for the annual company picnic this Saturday.", "safe"),
    ("Your recent transaction of Rs 500 at grocery store was successful.", "safe"),
    ("Thank you for contacting support your ticket number is 45678.", "safe"),
]

def extract_features(text):
    features = {}
    urls = re.findall(r'http[s]?://\S+', text)
    features['url_count'] = len(urls)
    suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.pw']
    features['suspicious_url'] = int(any(any(tld in url for tld in suspicious_tlds) for url in urls))
    urgency_words = ['urgent', 'immediately', 'suspend', 'verify', 'alert',
                     'warning', 'expire', 'limited', 'act now', 'click here', 'confirm']
    text_lower = text.lower()
    features['urgency_score'] = sum(1 for w in urgency_words if w in text_lower)
    prize_words = ['winner', 'won', 'prize', 'free', 'gift', 'reward', 'cash', 'million', 'lottery']
    features['prize_score'] = sum(1 for w in prize_words if w in text_lower)
    threat_words = ['suspended', 'blocked', 'deleted', 'terminated', 'locked', 'compromised', 'breach']
    features['threat_score'] = sum(1 for w in threat_words if w in text_lower)
    upper = sum(1 for c in text if c.isupper())
    features['upper_ratio'] = upper / max(len(text), 1)
    features['exclamation_count'] = text.count('!')
    features['text_length'] = len(text)
    features['has_numbers'] = int(bool(re.search(r'\d+', text)))
    return features

print("=" * 55)
print("   PHISHING EMAIL DETECTION MODEL")
print("   Thiranex Cybersecurity Internship")
print("=" * 55)

df = pd.DataFrame(emails_data, columns=['text', 'label'])
print(f"\n✅ Dataset: {len(df)} emails | Phishing: {len(df[df.label=='phishing'])} | Safe: {len(df[df.label=='safe'])}")

feature_df = pd.DataFrame(list(df['text'].apply(extract_features)))
le = LabelEncoder()
y = le.fit_transform(df['label'])

X_text_train, X_text_test, X_feat_train, X_feat_test, y_train, y_test = \
    train_test_split(df['text'], feature_df.values, y, test_size=0.25, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,2), stop_words='english')
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test  = tfidf.transform(X_text_test)

X_train = hstack([X_tfidf_train, csr_matrix(X_feat_train)])
X_test  = hstack([X_tfidf_test,  csr_matrix(X_feat_test)])

models = {
    "Random Forest"      : RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes"        : MultinomialNB(),
}

print("\n" + "=" * 55)
print("   MODEL RESULTS")
print("=" * 55)

best_model, best_acc, best_name = None, 0, ""
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"  {name}: {acc*100:.1f}% accuracy")
    if acc > best_acc:
        best_acc, best_model, best_name = acc, model, name

print(f"\n🏆 Best Model: {best_name} ({best_acc*100:.1f}%)")

y_pred = best_model.predict(X_test)
print("\n  Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Phishing','Safe']))

cm = confusion_matrix(y_test, y_pred)
print("  Confusion Matrix:")
print(f"                Predicted")
print(f"                Phishing  Safe")
print(f"  Actual Phishing   {cm[0][0]}        {cm[0][1]}")
print(f"  Actual Safe       {cm[1][0]}        {cm[1][1]}")

print("\n" + "=" * 55)
print("   LIVE PREDICTIONS")
print("=" * 55)

test_emails = [
    "URGENT! Your account suspended. Verify: http://paypal-verify.tk/login",
    "Hi Sarah, please find the project report attached.",
    "You won $500 gift card! Claim: http://free-reward.ml/get",
    "Your order has been shipped. Track at amazon.com",
    "Security alert: Secure account now: http://google-secure.ga",
]

for email in test_emails:
    feat = np.array(list(extract_features(email).values())).reshape(1,-1)
    vec  = hstack([tfidf.transform([email]), csr_matrix(feat)])
    pred = best_model.predict(vec)[0]
    conf = max(best_model.predict_proba(vec)[0]) * 100
    label = le.inverse_transform([pred])[0]
    icon = "🚨 PHISHING" if label == "phishing" else "✅ SAFE"
    print(f"\n  {email[:55]}...")
    print(f"  → {icon} | Confidence: {conf:.1f}%")

print("\n" + "=" * 55)
print("  Done! Thiranex Cybersecurity Internship")
print("=" * 55)

   PHISHING EMAIL DETECTION MODEL
   Thiranex Cybersecurity Internship

✅ Dataset: 40 emails | Phishing: 20 | Safe: 20

   MODEL RESULTS
  Random Forest: 100.0% accuracy
  Logistic Regression: 90.0% accuracy
  Naive Bayes: 50.0% accuracy

🏆 Best Model: Random Forest (100.0%)

  Classification Report:
              precision    recall  f1-score   support

    Phishing       1.00      1.00      1.00         5
        Safe       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

  Confusion Matrix:
                Predicted
                Phishing  Safe
  Actual Phishing   5        0
  Actual Safe       0        5

   LIVE PREDICTIONS

  URGENT! Your account suspended. Verify: http://paypal-v...
  → 🚨 PHISHING | Confidence: 87.0%

  Hi Sarah, please find the project report attached....
  → ✅ SAFE | Confidence: 93.0%

  You won $500 gift card!